# Reproduce Identity2Vec (I2V) — 3-seed evaluation

Run top to bottom (**Shift+Enter**). You edit **two lines** in Step 1: `DATASET` and `SEEDS`.

The pipeline: pick a dataset (auto-aligned to a safe graph+labels version if needed) → train a fresh embedding **per seed** → score **node classification** (micro/macro/weighted F1) and **link prediction** (AUC) → report **mean ± std** across seeds.

All heavy logic lives in the project files (`make_labels.py`, `scripts/runner.py`); the notebook only sets knobs and calls them.


## Step 0 — Set up

Tells Python where the project is. Run once.


In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
for p in (str(ROOT), str(ROOT / 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('Working from:', Path.cwd())

Working from: /home/m-adam/identity2vec


## Step 1 — Choose dataset + seeds ⬅️ the only place you edit

`prepare_dataset` resolves the dataset to a **safe** version (e.g. author `citeseer` → aligned `citeseer_linqs`, built from one LINQS source so graph node-ids and labels match). It prints what it used and why, and never touches the original author files.


In [ ]:
DATASET = "cora"        # cora | citeseer_linqs | pubmed | pubmed_linqs | amazon_computers | amazon_photo | coauthor_cs | coauthor_physics
SEEDS   = [42, 43, 44]      # 3 seeds for mean ± std
    
from make_labels import prepare_dataset
info = prepare_dataset(DATASET)   # -> {base, version, safe, edge_path, label_path}; builds aligned version if missing

Label file already exists: labels/cora.labels
dataset ready -> base=cora version=orig safe=cora
  edges  = input/cora.edgelist
  labels = labels/cora.labels


## Step 2 — Node classification (one embedding per seed)

For each seed: train a full-graph embedding → 70/30 stratified label split → logistic regression → **micro / macro / weighted F1**.
Embeddings are saved as `output/{base}_nc_{version}_s{seed}.emb`.


In [3]:
from runner import run_nodeclass_repeated
node_rows = run_nodeclass_repeated(info, seeds=SEEDS)

  [identity2vec nc s42] reuse existing cora_nc_orig_s42.emb
  [identity2vec nc s42] micro=0.7429 macro=0.7263 weighted=0.7417
  [identity2vec nc s43] reuse existing cora_nc_orig_s43.emb
  [identity2vec nc s43] micro=0.7528 macro=0.7419 weighted=0.7511
  [identity2vec nc s44] reuse existing cora_nc_orig_s44.emb
  [identity2vec nc s44] micro=0.7294 macro=0.7125 weighted=0.7280


## Step 3 — Link prediction (one split + embedding per seed)

For each seed: hide 30% of edges → train an embedding on the **70% train graph only** (leakage-free) → **AUC**.
Splits: `splits/{base}_lp_{version}_s{seed}_*`; embeddings: `output/{base}_lp_{version}_s{seed}.emb`.


In [4]:
from runner import run_linkpred_repeated
lp_rows = run_linkpred_repeated(info, seeds=SEEDS)

  [identity2vec lp s42] reuse existing cora_lp_orig_s42.emb
  [identity2vec lp s42] AUC=0.8305 (logreg=0.8226 cosine=0.8305)
  [identity2vec lp s43] reuse existing cora_lp_orig_s43.emb
  [identity2vec lp s43] AUC=0.8351 (logreg=0.8414 cosine=0.8351)
  [identity2vec lp s44] reuse existing cora_lp_orig_s44.emb
  [identity2vec lp s44] AUC=0.8187 (logreg=0.8367 cosine=0.8187)


## Step 4 — Results + summary (mean ± std)

Per-seed table, then the summary (`mean`, sample `std`, `delta = max − min`). Both saved to `results/`.


In [5]:
import os
from runner import summarize_seed_results

per_seed, summary = summarize_seed_results(node_rows, lp_rows)

tag = f"{info['base']}_{info['version']}"
res_dir = f"results/{info['safe']}"          # per-dataset subfolder
os.makedirs(res_dir, exist_ok=True)
per_seed.to_csv(f"{res_dir}/{tag}_per_seed.csv", index=False)
summary.to_csv(f"{res_dir}/{tag}_summary.csv", index=False)

def make_clean_table(per_seed, summary, task):
    task_rows = per_seed[per_seed["task"] == task].copy()

    if task == "nodeclass":
        metrics = ["micro_f1", "macro_f1", "weighted_f1"]
    elif task == "linkpred":
        metrics = ["auc"]
    else:
        raise ValueError(f"Unknown task: {task}")

    long_rows = task_rows.melt(
        id_vars=["dataset", "version", "task", "seed"],
        value_vars=metrics,
        var_name="metric",
        value_name="score",
    ).dropna(subset=["score"])

    seed_table = long_rows.pivot_table(
        index=["dataset", "version", "task", "metric"],
        columns="seed",
        values="score",
        aggfunc="first",
    ).reset_index()

    seed_table = seed_table.rename(columns={
        42: "seed_42",
        43: "seed_43",
        44: "seed_44",
    })

    summary_task = summary[summary["task"] == task].copy()

    clean = seed_table.merge(
        summary_task[["dataset", "version", "task", "metric", "mean", "std", "delta"]],
        on=["dataset", "version", "task", "metric"],
        how="left",
    )

    clean["final_score"] = clean.apply(
        lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}",
        axis=1,
    )

    # metric 1st column, task 2nd column on both tables; rows numbered from 1 not 0
    lead = ["metric", "task"]
    clean = clean[lead + [c for c in clean.columns if c not in lead]]
    clean = clean.sort_values("metric").reset_index(drop=True)
    clean.index = range(1, len(clean) + 1)

    return clean

node_table = make_clean_table(per_seed, summary, "nodeclass")
lp_table = make_clean_table(per_seed, summary, "linkpred")

node_table.to_csv(f"{res_dir}/{tag}_nodeclass_table.csv", index=False)
lp_table.to_csv(f"{res_dir}/{tag}_linkpred_table.csv", index=False)

print("\n=== Node Classification Table ===")
display(node_table)

print("\n=== Link Prediction Table ===")
display(lp_table)


=== Node Classification Table ===


,metric,task,dataset,version,seed_42,seed_43,seed_44,mean,std,delta,final_score
1,macro_f1,nodeclass,cora,orig,0.726273,0.741914,0.712487,0.726891,0.014723,0.029427,0.7269 ± 0.0147
2,micro_f1,nodeclass,cora,orig,0.742927,0.752768,0.729397,0.741697,0.011734,0.023370,0.7417 ± 0.0117
3,weighted_f1,nodeclass,cora,orig,0.741744,0.751138,0.728042,0.740308,0.011615,0.023096,0.7403 ± 0.0116



=== Link Prediction Table ===


,metric,task,dataset,version,seed_42,seed_43,seed_44,mean,std,delta,final_score
1,auc,linkpred,cora,orig,0.830505,0.835078,0.818674,0.828086,0.008465,0.016404,0.8281 ± 0.0085


## Step 5 — Cross-model benchmark (I2V vs deepwalk / node2vec / struc2vec)

**Edit only the 3 lists at the top of the cell below** (`RUN_DATASETS`, `RUN_MODELS`, `RUN_SEEDS`), then run it. **Do NOT change Step 1's `DATASET`** — that variable is only for the single-dataset Steps 2–4.

Examples:

- Cora + Identity2Vec: `RUN_DATASETS=["cora"]`, `RUN_MODELS=["identity2vec"]`, `RUN_SEEDS=[42]`
- same dataset, DeepWalk: change one line → `RUN_MODELS=["deepwalk"]`
- full comparison: `RUN_DATASETS=["cora","citeseer_linqs","enzymes","webkb_wisc"]`, `RUN_MODELS=["identity2vec","deepwalk","node2vec","struc2vec"]`, `RUN_SEEDS=[42,43,44]`

Outputs two tables (rows = datasets, columns = methods): **Table 1** node-class weighted F1 (mean ± std), **Table 2** link-pred AUC (mean ± std). Embeddings + splits are cached by filename, so re-runs train only what's missing. Saved under `results/benchmark/`. _(politics was dropped — rt-pol has no verifiable node labels; webkb_wisc replaces it, fully labelled with 5 classes.)_


In [6]:
# Step 5 — Cross-model benchmark.
# EDIT ONLY these 3 lists to choose what runs.

RUN_DATASETS = ["cora"]
RUN_MODELS   = ["identity2vec", "deepwalk", "node2vec", "struc2vec"]
RUN_SEEDS    = [42, 43, 44]

from benchmark_baselines import run_benchmark, save_benchmark


def clean_table_for_display(table, dataset_order=None, model_order=None):
    """Remove Pandas index/column names and show Dataset as a normal column."""
    table = table.copy()

    if table.empty:
        return table

    table.index.name = None
    table.columns.name = None

    table = table.reset_index()
    table = table.rename(columns={table.columns[0]: "Dataset"})

    if dataset_order is not None and "Dataset" in table.columns:
        table["_dataset_order"] = table["Dataset"].apply(
            lambda x: dataset_order.index(x) if x in dataset_order else len(dataset_order)
        )
        table = table.sort_values("_dataset_order").drop(columns=["_dataset_order"])

    if model_order is not None:
        ordered_cols = ["Dataset"]
        ordered_cols += [m for m in model_order if m in table.columns]
        ordered_cols += [c for c in table.columns if c not in ordered_cols]
        table = table[ordered_cols]

    return table.reset_index(drop=True)


# Show per-seed training progress, then the final tables.
bench_rows = run_benchmark(
    datasets=RUN_DATASETS,
    models=RUN_MODELS,
    seeds=RUN_SEEDS,
)

nc_table, lp_table = save_benchmark(bench_rows)


nc_show = clean_table_for_display(
    nc_table,
    dataset_order=RUN_DATASETS,
    model_order=RUN_MODELS,
)

lp_show = clean_table_for_display(
    lp_table,
    dataset_order=RUN_DATASETS,
    model_order=RUN_MODELS,
)

print("=== Table 1: Node Classification (Weighted F1, mean ± std) ===")
display(nc_show)

print("=== Table 2: Link Prediction (AUC, mean ± std) ===")
display(lp_show)

Label file already exists: labels/cora.labels
dataset ready -> base=cora version=orig safe=cora
  edges  = input/cora.edgelist
  labels = labels/cora.labels

=== identity2vec on cora ===
  [identity2vec nc s42] reuse existing cora_nc_orig_s42.emb
  [identity2vec nc s42] micro=0.7429 macro=0.7263 weighted=0.7417
  [identity2vec nc s43] reuse existing cora_nc_orig_s43.emb
  [identity2vec nc s43] micro=0.7528 macro=0.7419 weighted=0.7511
  [identity2vec nc s44] reuse existing cora_nc_orig_s44.emb
  [identity2vec nc s44] micro=0.7294 macro=0.7125 weighted=0.7280
  [identity2vec lp s42] reuse existing cora_lp_orig_s42.emb
  [identity2vec lp s42] AUC=0.8305 (logreg=0.8226 cosine=0.8305)
  [identity2vec lp s43] reuse existing cora_lp_orig_s43.emb
  [identity2vec lp s43] AUC=0.8351 (logreg=0.8414 cosine=0.8351)
  [identity2vec lp s44] reuse existing cora_lp_orig_s44.emb
  [identity2vec lp s44] AUC=0.8187 (logreg=0.8367 cosine=0.8187)

=== deepwalk on cora ===
  [deepwalk nc s42] reuse existing

,Dataset,identity2vec,deepwalk,node2vec,struc2vec
0,cora,0.7403 ± 0.0116,0.8109 ± 0.0216,0.8166 ± 0.0090,0.3219 ± 0.0053


=== Table 2: Link Prediction (AUC, mean ± std) ===


,Dataset,identity2vec,deepwalk,node2vec,struc2vec
0,cora,0.8281 ± 0.0085,0.9011 ± 0.0017,0.9031 ± 0.0029,0.5491 ± 0.0077


In [7]:
import pandas as pd  # add if not already imported in this cell

# ============================================================
# Original I2V (paper) reference results — for side-by-side comparison.
# Column order = RUN_MODELS, so these line up with your reproduced tables.
# Paper reports single numbers (no std); "____" = fill in yourself.
# ============================================================

# Link Prediction — paper Table 4 (AUC). Cora filled from the table you sent.
PAPER_LP = {
    "cora": {"identity2vec": 0.8413, "deepwalk": 0.7529, "node2vec": 0.7658, "struc2vec": 0.7115},
    # Other paper Table 4 rows (uncomment when you add them to RUN_DATASETS):
    # "citeseer_linqs": {"identity2vec": 0.8373, "deepwalk": 0.7301, "node2vec": 0.7951, "struc2vec": 0.6962},  # NOTE: paper's Citeseer is a DIFFERENT graph than citeseer_linqs
    # "enzymes":        {"identity2vec": 0.8024, "deepwalk": 0.7248, "node2vec": 0.7419, "struc2vec": 0.6902},
    # webkb_wisc: NOT in the paper's Table 4 (webkb only appears in Figure 2) -> stays "____".
    # LINE (paper, not in your models): Cora = 0.5407, if you ever add it.
}

# Node Classification — paper reports a PLOT (Figure 5, F1 vs train-ratio), not a table.
# Read values off the figure at your train ratio and fill these in.
PAPER_NC = {
    "cora": {"identity2vec": 0.7345, "deepwalk": 0.6162, "node2vec": 0.6384, "struc2vec": 0.6912},
}

# Build a paper table in the same Dataset x models shape ("____" where unfilled).
def paper_table(values, datasets, models):
    """Turn the manual {dataset: {model: value}} dict into a display table."""
    def cell(d, m):
        v = values.get(d, {}).get(m)
        return f"{v:.4f}" if isinstance(v, (int, float)) else "____"
    rows = [{"Dataset": d, **{m: cell(d, m) for m in models}} for d in datasets]
    return pd.DataFrame(rows)[["Dataset"] + models]

print("=== Table 1: Node Classification (Weighted F1, mean ± std) ===")
display(nc_show)

print("=== Original I2V Results — Node Classification (paper, Figure 5) ===")
display(paper_table(PAPER_NC, RUN_DATASETS, RUN_MODELS))

print("=== Table 2: Link Prediction (AUC, mean ± std) ===")
display(lp_show)

print("=== Original I2V Results — Link Prediction (paper, Table 4) ===")
display(paper_table(PAPER_LP, RUN_DATASETS, RUN_MODELS))

=== Table 1: Node Classification (Weighted F1, mean ± std) ===


,Dataset,identity2vec,deepwalk,node2vec,struc2vec
0,cora,0.7403 ± 0.0116,0.8109 ± 0.0216,0.8166 ± 0.0090,0.3219 ± 0.0053


=== Original I2V Results — Node Classification (paper, Figure 5) ===


,Dataset,identity2vec,deepwalk,node2vec,struc2vec
0,cora,0.7345,0.6162,0.6384,0.6912


=== Table 2: Link Prediction (AUC, mean ± std) ===


,Dataset,identity2vec,deepwalk,node2vec,struc2vec
0,cora,0.8281 ± 0.0085,0.9011 ± 0.0017,0.9031 ± 0.0029,0.5491 ± 0.0077


=== Original I2V Results — Link Prediction (paper, Table 4) ===


,Dataset,identity2vec,deepwalk,node2vec,struc2vec
0,cora,0.8413,0.7529,0.7658,0.7115


In [ ]:
## Step 6 — Dataset-wise comparison from saved results + existing files only
# This cell does NOT train anything.
# It shows only one final table.
# It checks:
# 1) results/benchmark/benchmark_per_seed.csv
# 2) old Steps 2–4 I2V result files
# 3) output/ and splits/ files already present on disk

from pathlib import Path
from contextlib import redirect_stdout
import io
import pandas as pd

from make_labels import prepare_dataset


COMPARE_DATASETS = ["cora"]   # example: ["cora", "citeseer_linqs", "enzymes", "webkb_wisc"]
COMPARE_MODELS   = ["identity2vec", "deepwalk", "node2vec", "struc2vec"]
COMPARE_SEEDS    = [42, 43, 44]


def quiet_prepare_dataset(dataset):
    silent = io.StringIO()
    with redirect_stdout(silent):
        info = prepare_dataset(dataset)
    return info


def model_prefix(model):
    if model == "identity2vec":
        return ""
    return f"{model}_"


def candidate_output_dirs(info):
    return [
        Path("output") / info["safe"],
        Path("output") / info["base"],
        Path("output"),
    ]


def candidate_split_dirs(info):
    return [
        Path("splits") / info["safe"],
        Path("splits") / info["base"],
        Path("splits"),
    ]


def embedding_exists(info, model, task_short, seed):
    prefix = model_prefix(model)
    fname = f"{prefix}{info['base']}_{task_short}_{info['version']}_s{seed}.emb"

    for out_dir in candidate_output_dirs(info):
        if (out_dir / fname).exists():
            return True

    return False


def split_exists(info, seed):
    stem = f"{info['base']}_lp_{info['version']}_s{seed}"

    for split_dir in candidate_split_dirs(info):
        train_file = split_dir / f"{stem}_train.edgelist"
        test_pos = list(split_dir.glob(f"{stem}_test_pos*"))
        test_neg = list(split_dir.glob(f"{stem}_test_neg*"))

        if train_file.exists() and len(test_pos) > 0 and len(test_neg) > 0:
            return True

    return False


def normalize_results_columns(df):
    df = df.copy()

    if "method" in df.columns and "model" not in df.columns:
        df = df.rename(columns={"method": "model"})

    if "dataset_name" in df.columns and "dataset" not in df.columns:
        df = df.rename(columns={"dataset_name": "dataset"})

    if "weighted" in df.columns and "weighted_f1" not in df.columns:
        df = df.rename(columns={"weighted": "weighted_f1"})

    if "roc_auc" in df.columns and "auc" not in df.columns:
        df = df.rename(columns={"roc_auc": "auc"})

    if "task" in df.columns:
        df["task"] = df["task"].replace({
            "nc": "nodeclass",
            "node_classification": "nodeclass",
            "nodeclassification": "nodeclass",
            "lp": "linkpred",
            "link_prediction": "linkpred",
            "linkprediction": "linkpred",
        })

    return df


def load_saved_results(datasets):
    frames = []

    bench_file = Path("results/benchmark/benchmark_per_seed.csv")
    if bench_file.exists():
        df = pd.read_csv(bench_file)
        df = normalize_results_columns(df)
        frames.append(df)

    for dataset in datasets:
        info = quiet_prepare_dataset(dataset)

        old_file = Path("results") / info["safe"] / f"{info['base']}_{info['version']}_per_seed.csv"

        if old_file.exists():
            df = pd.read_csv(old_file)
            df = normalize_results_columns(df)

            # Old Steps 2–4 are Identity2Vec-only.
            df["dataset"] = info["safe"]
            df["model"] = "identity2vec"

            frames.append(df)

    if len(frames) == 0:
        return pd.DataFrame()

    results = pd.concat(frames, ignore_index=True)
    results = normalize_results_columns(results)

    needed = ["dataset", "model", "task", "seed"]
    if all(col in results.columns for col in needed):
        results = results.drop_duplicates(
            subset=["dataset", "model", "task", "seed"],
            keep="last",
        )

    return results


def saved_score_or_none(results, dataset_names, model, task, metric):
    if results.empty:
        return None

    required_cols = ["dataset", "model", "task", "seed", metric]
    if not all(col in results.columns for col in required_cols):
        return None

    rows = results[
        (results["dataset"].isin(dataset_names))
        & (results["model"] == model)
        & (results["task"] == task)
        & (results["seed"].isin(COMPARE_SEEDS))
        & (results[metric].notna())
    ].copy()

    done_seeds = sorted(rows["seed"].unique().tolist())

    if len(done_seeds) == 0:
        return None

    mean = rows[metric].mean()
    std = rows[metric].std(ddof=1) if len(rows) > 1 else 0.0

    if len(done_seeds) < len(COMPARE_SEEDS):
        return f"{mean:.4f} ± {std:.4f} ({len(done_seeds)}/{len(COMPARE_SEEDS)} seeds)"

    return f"{mean:.4f} ± {std:.4f}"


def file_status(info, model, task):
    if task == "nodeclass":
        done = [
            seed for seed in COMPARE_SEEDS
            if embedding_exists(info, model, "nc", seed)
        ]

        if len(done) == len(COMPARE_SEEDS):
            return "FILES READY, SCORE MISSING"

        return f"WAIT: NC embeddings {len(done)}/{len(COMPARE_SEEDS)}"

    if task == "linkpred":
        done = [
            seed for seed in COMPARE_SEEDS
            if embedding_exists(info, model, "lp", seed) and split_exists(info, seed)
        ]

        if len(done) == len(COMPARE_SEEDS):
            return "FILES READY, SCORE MISSING"

        return f"WAIT: LP files {len(done)}/{len(COMPARE_SEEDS)}"

    raise ValueError(f"Unknown task: {task}")


results = load_saved_results(COMPARE_DATASETS)

all_rows = []

for dataset in COMPARE_DATASETS:
    info = quiet_prepare_dataset(dataset)

    dataset_names = list({
        dataset,
        info["safe"],
        info["base"],
    })

    for model in COMPARE_MODELS:
        nc_score = saved_score_or_none(
            results=results,
            dataset_names=dataset_names,
            model=model,
            task="nodeclass",
            metric="weighted_f1",
        )

        lp_score = saved_score_or_none(
            results=results,
            dataset_names=dataset_names,
            model=model,
            task="linkpred",
            metric="auc",
        )

        all_rows.append({
            "Dataset": info["safe"],
            "Model": model,
            "NC weighted F1": nc_score if nc_score is not None else file_status(info, model, "nodeclass"),
            "LP AUC": lp_score if lp_score is not None else file_status(info, model, "linkpred"),
        })

compare_table = pd.DataFrame(all_rows)

display(compare_table)

,Dataset,Model,NC weighted F1,LP AUC
0,cora,identity2vec,0.7403 ± 0.0116,0.8281 ± 0.0085
1,cora,deepwalk,0.8109 ± 0.0216,0.9011 ± 0.0017
2,cora,node2vec,0.8166 ± 0.0090,0.9031 ± 0.0029
3,cora,struc2vec,0.3219 ± 0.0053,0.5491 ± 0.0077


In [ ]:
# ## Step 7A — Result completeness sanity check
# # Put after Step 6.
# # This does NOT train anything.
# # It checks benchmark_per_seed.csv for missing/duplicate model-seed-task rows.

# from pathlib import Path
# import pandas as pd

# CHECK_DATASET = "cora"
# CHECK_MODELS = ["identity2vec", "deepwalk", "node2vec", "struc2vec"]
# CHECK_SEEDS = [42, 43, 44]

# bench_file = Path("results/benchmark/benchmark_per_seed.csv")

# if not bench_file.exists():
#     raise FileNotFoundError("results/benchmark/benchmark_per_seed.csv not found. Run Step 5 first.")

# df = pd.read_csv(bench_file)

# if "method" in df.columns and "model" not in df.columns:
#     df = df.rename(columns={"method": "model"})

# if "roc_auc" in df.columns and "auc" not in df.columns:
#     df = df.rename(columns={"roc_auc": "auc"})

# if "weighted" in df.columns and "weighted_f1" not in df.columns:
#     df = df.rename(columns={"weighted": "weighted_f1"})

# df = df[df["dataset"].isin([CHECK_DATASET])].copy()

# summary_rows = []

# for model in CHECK_MODELS:
#     for task, metric in [("nodeclass", "weighted_f1"), ("linkpred", "auc")]:
#         rows = df[
#             (df["model"] == model)
#             & (df["task"] == task)
#             & (df["seed"].isin(CHECK_SEEDS))
#         ].copy()

#         scored_rows = rows[rows[metric].notna()] if metric in rows.columns else pd.DataFrame()
#         done_seeds = sorted(scored_rows["seed"].unique().tolist()) if not scored_rows.empty else []
#         missing_seeds = [s for s in CHECK_SEEDS if s not in done_seeds]

#         mean = scored_rows[metric].mean() if len(scored_rows) else None
#         std = scored_rows[metric].std(ddof=1) if len(scored_rows) > 1 else 0.0 if len(scored_rows) == 1 else None

#         summary_rows.append({
#             "Dataset": CHECK_DATASET,
#             "Model": model,
#             "Task": task,
#             "Metric": metric,
#             "Done seeds": done_seeds,
#             "Missing seeds": missing_seeds,
#             "Mean": None if mean is None else round(mean, 4),
#             "Std": None if std is None else round(std, 4),
#         })

# summary_table = pd.DataFrame(summary_rows)
# display(summary_table)

# duplicates = df[df.duplicated(subset=["dataset", "model", "task", "seed"], keep=False)]

# if len(duplicates) > 0:
#     display(duplicates.sort_values(["model", "task", "seed"]))

In [ ]:
# ## Step 7B — Link-pred split difficulty / proximity diagnostic
# # Put after Step 7A.
# # This does NOT train anything.
# # It scores the SAME LP test edges using simple graph heuristics.

# from pathlib import Path
# import math
# import networkx as nx
# import pandas as pd
# from sklearn.metrics import roc_auc_score
# from contextlib import redirect_stdout
# import io

# from make_labels import prepare_dataset

# CHECK_DATASET = "cora"
# CHECK_SEEDS = [42, 43, 44]


# def quiet_prepare_dataset(dataset):
#     silent = io.StringIO()
#     with redirect_stdout(silent):
#         return prepare_dataset(dataset)


# def candidate_split_dirs(info):
#     return [
#         Path("splits") / info["safe"],
#         Path("splits") / info["base"],
#         Path("splits"),
#     ]


# def find_split_files(info, seed):
#     stem = f"{info['base']}_lp_{info['version']}_s{seed}"

#     for split_dir in candidate_split_dirs(info):
#         train_file = split_dir / f"{stem}_train.edgelist"
#         pos_files = list(split_dir.glob(f"{stem}_test_pos*"))
#         neg_files = list(split_dir.glob(f"{stem}_test_neg*"))

#         if train_file.exists() and len(pos_files) > 0 and len(neg_files) > 0:
#             return train_file, pos_files[0], neg_files[0]

#     raise FileNotFoundError(f"Split files not found for seed={seed}, stem={stem}")


# def read_edge_file(path):
#     edges = []

#     with open(path, "r", encoding="utf-8") as f:
#         for line in f:
#             parts = line.strip().split()
#             if len(parts) >= 2:
#                 edges.append((str(parts[0]), str(parts[1])))

#     return edges


# def common_neighbors_score(G, u, v):
#     if not G.has_node(u) or not G.has_node(v) or u == v:
#         return 0.0
#     return float(len(list(nx.common_neighbors(G, u, v))))


# def jaccard_score(G, u, v):
#     if not G.has_node(u) or not G.has_node(v) or u == v:
#         return 0.0

#     nu = set(G.neighbors(u))
#     nv = set(G.neighbors(v))

#     union = nu | nv
#     if len(union) == 0:
#         return 0.0

#     return len(nu & nv) / len(union)


# def adamic_adar_score(G, u, v):
#     if not G.has_node(u) or not G.has_node(v) or u == v:
#         return 0.0

#     score = 0.0

#     for w in nx.common_neighbors(G, u, v):
#         deg = G.degree(w)
#         if deg > 1:
#             score += 1.0 / math.log(deg)

#     return score


# info = quiet_prepare_dataset(CHECK_DATASET)

# rows = []

# for seed in CHECK_SEEDS:
#     train_file, pos_file, neg_file = find_split_files(info, seed)

#     G_train = nx.read_edgelist(train_file, nodetype=str)
#     pos_edges = read_edge_file(pos_file)
#     neg_edges = read_edge_file(neg_file)

#     pairs = pos_edges + neg_edges
#     y_true = [1] * len(pos_edges) + [0] * len(neg_edges)

#     cn_scores = [common_neighbors_score(G_train, u, v) for u, v in pairs]
#     jc_scores = [jaccard_score(G_train, u, v) for u, v in pairs]
#     aa_scores = [adamic_adar_score(G_train, u, v) for u, v in pairs]

#     rows.append({
#         "Dataset": info["safe"],
#         "Seed": seed,
#         "Train edges": G_train.number_of_edges(),
#         "Test positive": len(pos_edges),
#         "Test negative": len(neg_edges),
#         "Common Neighbors AUC": round(roc_auc_score(y_true, cn_scores), 4),
#         "Jaccard AUC": round(roc_auc_score(y_true, jc_scores), 4),
#         "Adamic-Adar AUC": round(roc_auc_score(y_true, aa_scores), 4),
#     })

# proximity_table = pd.DataFrame(rows)
# display(proximity_table)

# display(
#     proximity_table[
#         ["Common Neighbors AUC", "Jaccard AUC", "Adamic-Adar AUC"]
#     ].mean().round(4).to_frame("Mean over seeds")
# )

In [ ]:
# ## Step 7C — Embedding separation diagnostic
# # Put after Step 7B.
# # This does NOT train anything.
# # It reads existing LP embeddings and checks real-edge vs fake-edge cosine separation.

# from pathlib import Path
# import numpy as np
# import pandas as pd
# from sklearn.metrics import roc_auc_score

# CHECK_DATASET = "cora"
# CHECK_MODELS = ["identity2vec", "deepwalk", "node2vec", "struc2vec"]
# CHECK_SEEDS = [42, 43, 44]


# def model_prefix(model):
#     return "" if model == "identity2vec" else f"{model}_"


# def candidate_output_dirs(info):
#     return [
#         Path("output") / info["safe"],
#         Path("output") / info["base"],
#         Path("output"),
#     ]


# def find_embedding_file(info, model, task_short, seed):
#     prefix = model_prefix(model)
#     fname = f"{prefix}{info['base']}_{task_short}_{info['version']}_s{seed}.emb"

#     for out_dir in candidate_output_dirs(info):
#         path = out_dir / fname
#         if path.exists():
#             return path

#     return None


# def load_word2vec_embedding(path):
#     emb = {}

#     with open(path, "r", encoding="utf-8") as f:
#         first = f.readline().strip().split()

#         # Word2Vec text format usually starts with: num_nodes dim
#         if len(first) == 2 and first[0].isdigit() and first[1].isdigit():
#             pass
#         else:
#             node = first[0]
#             vec = np.array([float(x) for x in first[1:]], dtype=np.float32)
#             emb[node] = vec

#         for line in f:
#             parts = line.strip().split()
#             if len(parts) <= 2:
#                 continue

#             node = parts[0]
#             vec = np.array([float(x) for x in parts[1:]], dtype=np.float32)
#             emb[node] = vec

#     return emb


# def cosine_from_emb(emb, u, v):
#     if u not in emb or v not in emb:
#         return None

#     a = emb[u]
#     b = emb[v]

#     denom = np.linalg.norm(a) * np.linalg.norm(b)
#     if denom == 0:
#         return None

#     return float(np.dot(a, b) / denom)


# info = quiet_prepare_dataset(CHECK_DATASET)

# rows = []

# for model in CHECK_MODELS:
#     for seed in CHECK_SEEDS:
#         emb_file = find_embedding_file(info, model, "lp", seed)

#         if emb_file is None:
#             rows.append({
#                 "Model": model,
#                 "Seed": seed,
#                 "Status": "LP embedding missing",
#             })
#             continue

#         train_file, pos_file, neg_file = find_split_files(info, seed)

#         pos_edges = read_edge_file(pos_file)
#         neg_edges = read_edge_file(neg_file)

#         emb = load_word2vec_embedding(emb_file)

#         y_true = []
#         scores = []
#         pos_scores = []
#         neg_scores = []

#         for u, v in pos_edges:
#             score = cosine_from_emb(emb, str(u), str(v))
#             if score is not None:
#                 y_true.append(1)
#                 scores.append(score)
#                 pos_scores.append(score)

#         for u, v in neg_edges:
#             score = cosine_from_emb(emb, str(u), str(v))
#             if score is not None:
#                 y_true.append(0)
#                 scores.append(score)
#                 neg_scores.append(score)

#         if len(set(y_true)) < 2:
#             rows.append({
#                 "Model": model,
#                 "Seed": seed,
#                 "Status": "Not enough scored edges",
#             })
#             continue

#         auc = roc_auc_score(y_true, scores)
#         pos_mean = float(np.mean(pos_scores))
#         neg_mean = float(np.mean(neg_scores))
#         gap = pos_mean - neg_mean

#         rows.append({
#             "Model": model,
#             "Seed": seed,
#             "Status": "OK",
#             "Embedding nodes": len(emb),
#             "Scored pos": len(pos_scores),
#             "Scored neg": len(neg_scores),
#             "Cosine AUC": round(auc, 4),
#             "Mean real-edge cosine": round(pos_mean, 4),
#             "Mean fake-edge cosine": round(neg_mean, 4),
#             "Real-fake gap": round(gap, 4),
#             "Embedding file": str(emb_file),
#         })

# edge_sep = pd.DataFrame(rows)
# display(edge_sep)

# ok_rows = edge_sep[edge_sep["Status"] == "OK"].copy()

# if len(ok_rows) > 0:
#     display(
#         ok_rows.groupby("Model")[[
#             "Cosine AUC",
#             "Mean real-edge cosine",
#             "Mean fake-edge cosine",
#             "Real-fake gap",
#         ]].mean().round(4).reset_index()
#     )